# Silver — physical_vendas_caixa

Desenvolvido por: Luiz Henrique Portácio

Lê a Bronze de `physical_vendas_caixa`, aplica as 5 regras técnicas
e grava em Delta particionado por `ano` e `mes`.

**Regras técnicas:**
1. `id_transacao` (UUID) não pode ser nulo nem duplicado (PK).
2. `id_loja` deve existir em `physical_lojas` (FK).
3. `valor_total_venda` deve ser > 0.
4. `tipo_pagamento` deve estar em ('Cartão', 'Dinheiro', 'Pix').
5. `dt_venda` particionada por `ano` e `mes` para consultas eficientes.

**Esta camada NÃO grava no SQL Server.**

In [0]:
%run "../utils/00_utils"

In [0]:
adls_options = get_adls_options()

from pyspark.sql.functions import (
    col, year, month, to_date, trim,
    when, lit, udf, count
)
from pyspark.sql.types import StringType

SILVER_VENDAS_CAIXA_PATH          = f"{SILVER_BASE_PATH}physical_vendas_caixa"
SILVER_QUARENTENA_VENDAS_CAIXA_PATH = f"{SQUAD_ROOT_PATH}silver/_quarentena_physical_vendas_caixa"
TIPOS_PAGAMENTO_VALIDOS            = ["Cartão", "Dinheiro", "Pix"]
SILVER_WRITE_MODE                  = "overwrite"

print(f"Destino Silver : {SILVER_VENDAS_CAIXA_PATH}")
print(f"Modo de escrita: {SILVER_WRITE_MODE}")

## Leitura da Bronze

In [0]:
df_bronze = read_delta(BRONZE_VENDAS_CAIXA_PATH, adls_options)
print(f"Registros lidos da Bronze: {df_bronze.count():,}")
display(df_bronze.limit(5))

In [0]:
from pyspark.sql.functions import abs, when, col
from pyspark.sql.functions import round as spark_round

# Carrega a Silver
df_itens = read_delta(SILVER_ITENS_VENDA_CAIXA_PATH, adls_options)

# Filtra só os inconsistentes
df_inc = df_itens.filter(col("flag_valor_inconsistente") == True)

# Calcula a diferença real
df_analise = df_inc.withColumn(
    "diferenca_abs",
    spark_round(abs(col("valor_calculado") - col("valor_total_item_original")), 2)
).withColumn(
    "pct_diferenca",
    spark_round(
        (col("diferenca_abs") / col("valor_total_item_original")) * 100, 1
    )
)

# Distribuição das diferenças
print("Distribuição das diferenças de valor:")
df_analise.select("diferenca_abs", "pct_diferenca").describe().show()

# Casos onde a diferença é pequena (arredondamento) vs grande (desconto real)
df_analise.groupBy(
    when(col("diferenca_abs") < 0.02, "arredondamento")
    .when(col("diferenca_abs") < 1.0, "diferenca_pequena")
    .when(col("pct_diferenca") > 5, "desconto_provavel")
    .otherwise("outro")
    .alias("tipo_diferenca")
).count().show()

## Verificação de pré-condição

In [0]:
verificar_destino_limpo(
    SILVER_VENDAS_CAIXA_PATH,
    adls_options,
    permitir_existente=(SILVER_WRITE_MODE == "overwrite"),
)

## Tipagem das colunas

In [0]:
from pyspark.sql.functions import expr

df_tipado = (
    df_bronze
    .withColumn("id_transacao",       col("id_transacao"))
    .withColumn("id_loja",            expr("TRY_CAST(id_loja AS BIGINT)"))
    .withColumn("id_caixa",           expr("TRY_CAST(id_caixa AS BIGINT)"))
    .withColumn("id_operador",        expr("TRY_CAST(id_operador AS BIGINT)"))
    .withColumn("dt_venda",           to_date(col("dt_venda")))
    .withColumn("valor_total_venda",  expr("TRY_CAST(valor_total_venda AS DECIMAL(10,2))"))
    .withColumn("cpf_cliente",        col("cpf_cliente").cast("string"))
    .withColumn("tipo_pagamento",     trim(col("tipo_pagamento")))
    .withColumn("ano",                year(col("dt_venda")))
    .withColumn("mes",                month(col("dt_venda")))
)

## Regra 1 — PK: id_transacao não nulo nem duplicado

In [0]:
total_antes_pk = df_tipado.count()

df_pk_valida = separar_quarentena_pk(
    df=df_tipado,
    coluna_pk="id_transacao",
    quarentena_path=SILVER_QUARENTENA_VENDAS_CAIXA_PATH,
    adls_options=adls_options,
)

registrar_metrica_dq(
    tabela="physical_vendas_caixa",
    regra="01_pk_id_transacao_nula_ou_duplicada",
    qtd_registros_afetados=total_antes_pk - df_pk_valida.count(),
    qtd_registros_total=total_antes_pk,
    adls_options=adls_options,
)
print(f"[R1] PK: {df_pk_valida.count():,} registros válidos.")

## Regra 2 — FK: id_loja deve existir em physical_lojas

In [0]:
df_lojas_ids = (
    read_delta(SILVER_LOJAS_PATH, adls_options)
    .select(col("id_loja").alias("id_loja_ref"))
    .distinct()
)

total_fk = df_pk_valida.count()

df_fk = (
    df_pk_valida
    .join(
        df_lojas_ids.withColumn("_loja_existe", lit(True)),
        df_pk_valida["id_loja"] == df_lojas_ids["id_loja_ref"],
        how="left",
    )
    .withColumn("flag_fk_invalido", col("_loja_existe").isNull())
    .drop("id_loja_ref", "_loja_existe")
)

qtd_fk_invalido = df_fk.filter(col("flag_fk_invalido")).count()
registrar_metrica_dq(
    tabela="physical_vendas_caixa",
    regra="02_fk_id_loja_invalido",
    qtd_registros_afetados=qtd_fk_invalido,
    qtd_registros_total=total_fk,
    adls_options=adls_options,
)
print(f"[R2] FK: {qtd_fk_invalido:,} transações com id_loja inválido.")

## Regra 3 — valor_total_venda > 0

In [0]:
total_valor = df_fk.count()

df_valor = df_fk.withColumn(
    "flag_valor_invalido",
    col("valor_total_venda").isNull() | (col("valor_total_venda") <= 0)
)

qtd_valor_invalido = df_valor.filter(col("flag_valor_invalido")).count()
registrar_metrica_dq(
    tabela="physical_vendas_caixa",
    regra="03_valor_total_venda_invalido",
    qtd_registros_afetados=qtd_valor_invalido,
    qtd_registros_total=total_valor,
    adls_options=adls_options,
)
print(f"[R3] Valor: {qtd_valor_invalido:,} transações com valor inválido (nulo ou <= 0).")

## Regra 4 — tipo_pagamento válido

In [0]:
total_pagamento = df_valor.count()

df_pagamento = df_valor.withColumn(
    "flag_tipo_pagamento_invalido",
    ~col("tipo_pagamento").isin(TIPOS_PAGAMENTO_VALIDOS)
    | col("tipo_pagamento").isNull()
)

qtd_pagamento_invalido = df_pagamento.filter(col("flag_tipo_pagamento_invalido")).count()
registrar_metrica_dq(
    tabela="physical_vendas_caixa",
    regra="04_tipo_pagamento_invalido",
    qtd_registros_afetados=qtd_pagamento_invalido,
    qtd_registros_total=total_pagamento,
    adls_options=adls_options,
)
print(f"[R4] Pagamento: {qtd_pagamento_invalido:,} transações com tipo inválido.")
print(f"     Tipos válidos: {TIPOS_PAGAMENTO_VALIDOS}")

## Regra 5 — Flag é_feriado (enriquecimento)

In [0]:
FERIADOS_NACIONAIS = [
    "2024-01-01","2024-03-29","2024-04-21","2024-05-01",
    "2024-05-30","2024-09-07","2024-10-12","2024-11-02",
    "2024-11-15","2024-11-20","2024-12-25",
    "2025-01-01","2025-04-18","2025-04-21","2025-05-01",
    "2025-06-19","2025-09-07","2025-10-12","2025-11-02",
    "2025-11-15","2025-11-20","2025-12-25",
    "2026-01-01","2026-04-03","2026-04-21","2026-05-01",
    "2026-06-04","2026-09-07","2026-10-12","2026-11-02",
    "2026-11-15","2026-11-20","2026-12-25",
]

df_completo = df_pagamento.withColumn(
    "e_feriado",
    col("dt_venda").cast("string").isin(FERIADOS_NACIONAIS)
)

print(f"[R5] Feriado: {df_completo.filter(col('e_feriado')).count():,} transações em feriados.")

## Metadados de auditoria + escrita em Delta

Gravado particionado por `ano` e `mes` (Regra 5 — eficiência analítica).

In [0]:
df_silver_final = adicionar_metadados_silver(df_completo)

(
    df_silver_final.write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(SILVER_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(SILVER_VENDAS_CAIXA_PATH)
)

print(f"[OK] {df_silver_final.count():,} linha(s) gravada(s) em '{SILVER_VENDAS_CAIXA_PATH}'.")
print(f"     Particionado por: ano, mes")

## Validação final

In [0]:
df_saved = (
    read_delta(SILVER_VENDAS_CAIXA_PATH, adls_options)
)
print(f"Total gravado na Silver: {df_saved.count():,}")
display(df_saved.limit(5))